In [ ]:
!pip install -q pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import zipfile, os
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/TTTN/Distrinet-CIC-IDS-2017.zip"
DATA_DIR = "/content/dataset"
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zf:
     zf.extractall(DATA_DIR)
print(os.listdir(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Bruteforce-Tuesday.parquet', 'Benign-Monday.parquet', 'DoS-Wednesday.parquet', 'Portscan-DDos-Botnet-Friday.parquet', 'Infiltration-Webattacks-Thursday.parquet']


In [ ]:
for f in parquet_files:
    print(f)

/content/dataset/Benign-Monday.parquet
/content/dataset/Bruteforce-Tuesday.parquet
/content/dataset/DoS-Wednesday.parquet
/content/dataset/Infiltration-Webattacks-Thursday.parquet
/content/dataset/Portscan-DDos-Botnet-Friday.parquet


In [ ]:
row_counts2 = (
    df.groupby("__source_file").size()
      .reindex([os.path.basename(f) for f in parquet_files])
      .reset_index(name="Số dòng")
      .rename(columns={"__source_file": "Tên file"})
)
row_counts2["Số dòng"] = row_counts2["Số dòng"].apply(lambda x: f"{x:,}")
row_counts2

,Tên file,Số dòng
0,Benign-Monday.parquet,"350,718"
1,Bruteforce-Tuesday.parquet,"307,071"
2,DoS-Wednesday.parquet,"477,871"
3,Infiltration-Webattacks-Thursday.parquet,"281,439"
4,Portscan-DDos-Botnet-Friday.parquet,"370,259"


### 1. Gộp các file parquet

In [ ]:
import glob
import pandas as pd

parquet_files = sorted(glob.glob(f"{DATA_DIR}/*.parquet"))
print(f"Tìm thấy {len(parquet_files)} file parquet:")
for f in parquet_files:
    print(" -", os.path.basename(f))

dfs = []
for f in parquet_files:
    df_part = pd.read_parquet(f)
    df_part["__source_file"] = os.path.basename(f)  # lưu vết nguồn gốc từng dòng
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
print("\nShape sau khi gộp:", df.shape)

Tìm thấy 5 file parquet:
 - Benign-Monday.parquet
 - Bruteforce-Tuesday.parquet
 - DoS-Wednesday.parquet
 - Infiltration-Webattacks-Thursday.parquet
 - Portscan-DDos-Botnet-Friday.parquet

Shape sau khi gộp: (1787358, 84)


### 2. Kiểm kê dataset

In [ ]:
n_rows, n_cols = df.shape
print(f"Số dòng (samples): {n_rows:,}")
print(f"Số cột (bao gồm nhãn + cột nguồn gốc): {n_cols}")
print(f"Số feature (không tính nhãn 'Label' và '__source_file'): {n_cols - 2}")

Số dòng (samples): 1,787,358
Số cột (bao gồm nhãn + cột nguồn gốc): 84
Số feature (không tính nhãn 'Label' và '__source_file'): 82


In [ ]:
# Kiểu dữ liệu theo từng cột
dtype_summary = df.dtypes.value_counts()
print("Số lượng cột theo kiểu dữ liệu:")
print(dtype_summary)

print("\nChi tiết dtype từng cột:")
display(df.dtypes.to_frame(name="dtype"))

Số lượng cột theo kiểu dữ liệu:
float32    43
int8       14
int32      13
int16       9
float64     2
object      2
int64       1
Name: count, dtype: int64

Chi tiết dtype từng cột:


,dtype
Protocol,int8
Flow Duration,int32
Total Fwd Packets,int32
Total Backward Packets,int32
Fwd Packets Length Total,float32
...,...
ICMP Code,int8
ICMP Type,int8
Total TCP Flow Time,int64
Label,object


In [ ]:
# Missing value theo từng cột
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("Không có giá trị missing (NaN) trong dataset.")
else:
    print(f"Có {len(missing)} cột chứa giá trị missing:")
    display(missing.to_frame(name="so_luong_missing"))

Không có giá trị missing (NaN) trong dataset.


In [ ]:
# Kiểm tra thêm giá trị vô cực (Inf) — thường xuất hiện ở các đặc trưng dạng tỉ lệ (Flow Bytes/s, Flow Packets/s...)
import numpy as np

num_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[num_cols]).sum()
inf_counts = inf_counts[inf_counts > 0]

if len(inf_counts) == 0:
    print("Không có giá trị vô cực (Inf) trong dataset.")
else:
    print("Các cột có giá trị Inf:")
    display(inf_counts.to_frame(name="so_luong_inf"))

Không có giá trị vô cực (Inf) trong dataset.


In [ ]:
# Cấu trúc dữ liệu tổng quan
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1787358 entries, 0 to 1787357
Data columns (total 84 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   Protocol                  int8   
 1   Flow Duration             int32  
 2   Total Fwd Packets         int32  
 3   Total Backward Packets    int32  
 4   Fwd Packets Length Total  float32
 5   Bwd Packets Length Total  float32
 6   Fwd Packet Length Max     float32
 7   Fwd Packet Length Min     float32
 8   Fwd Packet Length Mean    float32
 9   Fwd Packet Length Std     float32
 10  Bwd Packet Length Max     float32
 11  Bwd Packet Length Min     float32
 12  Bwd Packet Length Mean    float32
 13  Bwd Packet Length Std     float32
 14  Flow Bytes/s              float64
 15  Flow Packets/s            float64
 16  Flow IAT Mean             float32
 17  Flow IAT Std              float32
 18  Flow IAT Max              float32
 19  Flow IAT Min              float32
 20  Fwd IAT Total           

In [ ]:
pd.set_option("display.max_columns", None)
df.head(10)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd RST Flags,Bwd RST Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Avg Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Act Data Packets,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label,__source_file
0,0,119719148,231,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,1.929516,5.205180e+05,2274525.75,20694308.0,0.0,119719152.0,5.205180e+05,2274525.75,20694308.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,1.929516,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0.0,0.000000,0.000000,0.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,8721244.0,8761761.0,22509460.0,17.0,12685486.0,5296658.0,20694308.0,6499982.0,-1,-1,0,Benign,Benign-Monday.parquet
1,17,65511209,6,6,288.0,288.0,48.0,48.0,48.000000,0.000000,48.0,48.0,48.0,0.0,8.792388,0.183175,5.955564e+06,19258082.00,64004884.0,3.0,65511096.0,1.310222e+07,28462970.00,64004960.0,3.0,65511104.0,13102221.0,28462988.0,64004996.0,3.0,0,0,0,0,0,0,48,48,0.091587,0.091587,48.0,48.0,48.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,1.0,48.000000,48.000000,48.0,0,0,0,0,0,0,0,24,0,24,0,0,5,8,1506210.0,0.0,1506210.0,1506210.0,64004884.0,0.0,64004884.0,64004884.0,-1,-1,0,Benign,Benign-Monday.parquet
2,17,113976922,267,0,20447.0,0.0,153.0,37.0,76.580521,44.140625,0.0,0.0,0.0,0.0,179.395966,2.342579,4.284847e+05,3701805.25,48523116.0,0.0,113976920.0,4.284847e+05,3701805.25,48523116.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,2136,0,2.342580,0.000000,37.0,153.0,76.580521,44.140625,1948.394775,0,0,0,0,0,0,0,0,0.0,76.580521,76.580521,0.0,2044,26,4794,0,0,0,1,76,0,0,0,0,266,8,2745981.5,5491934.5,10983883.0,14.0,25498178.0,18833046.0,48523116.0,5463561.0,-1,-1,0,Benign,Benign-Monday.parquet
3,17,67037196,8,8,384.0,384.0,48.0,48.0,48.000000,0.000000,48.0,48.0,48.0,0.0,11.456326,0.238673,4.469146e+06,14271655.00,55956316.0,44916.0,66990996.0,9.570142e+06,20477636.00,56001260.0,992873.0,66991088.0,9570156.0,20478190.0,56002516.0,991683.0,0,0,0,0,0,0,64,64,0.119337,0.119337,48.0,48.0,48.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,1.0,48.000000,48.000000,48.0,0,0,0,0,0,0,0,24,0,24,0,0,7,8,11034681.0,0.0,11034681.0,11034681.0,55956316.0,0.0,55956316.0,55956316.0,-1,-1,0,Benign,Benign-Monday.parquet
4,17,68045057,8,8,384.0,384.0,48.0,48.0,48.000000,0.000000,48.0,48.0,48.0,0.0,11.286639,0.235138,4.536337e+06,14525054.00,56943904.0,57451.0,67987504.0,9.712500e+06,20855822.00,57001356.0,989295.0,67987528.0,9712504.0,20855866.0,57001460.0,989348.0,0,0,0,0,0,0,64,64,0.117569,0.117569,48.0,48.0,48.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,1.0,48.000000,48.000000,48.0,0,0,0,0,0,0,0,24,0,24,0,0,7,8,11043596.0,0.0,11043596.0,11043596.0,56943904.0,0.0,56943904.0,56943904.0,-1,-1,0,Benign,Benign-Monday.parquet
5,17,70053281,3,3,144.0,144.0,48.0,48.0,48.000000,0.000000,48.0,48.0,48.0,0.0,4.111156,0.085649,1.401066e+07,29592712.00,66904116.0,93919.0,69959072.0,3.497954e+07,45281160.00,66998152.0,2960924.0,6995936

### 3. Thống kê nhãn gốc (trước khi gán nhãn nhị phân)

In [ ]:
label_counts = df["Label"].value_counts()
print("Phân bố nhãn gốc:")
display(label_counts.to_frame(name="so_luong"))

Phân bố nhãn gốc:


,so_luong
Label,
Benign,1496505
DoS Hulk,158449
DDoS,95144
Attempted-relabel-as-Benign,8966
DoS GoldenEye,7567
Infiltration - Portscan,5485
DoS Slowloris,3998
FTP-Patator,3972
SSH-Patator,2961


### 4. Gán nhãn nhị phân & loại bỏ nhãn "Attempted"


In [ ]:

mask_attempted = df["Label"].str.contains("attempted", case=False, na=False)
n_dropped = mask_attempted.sum()

df_labeled = df.loc[~mask_attempted].copy()

print(f"Số dòng bị loại vì nhãn chứa 'Attempted': {n_dropped:,}")
print(f"Số dòng còn lại: {len(df_labeled):,}")

Số dòng bị loại vì nhãn chứa 'Attempted': 8,966
Số dòng còn lại: 1,778,392


In [ ]:
# Gán nhãn nhị phân
df_labeled["Label_binary"] = (
    df_labeled["Label"].str.strip().str.lower() != "benign"
).astype("int8")

print("Phân bố nhãn nhị phân sau khi gán:")
display(df_labeled["Label_binary"].value_counts().rename({0: "Benign (0)", 1: "Attack (1)"}))

print("\nKiểm tra chéo Label gốc vs Label_binary:")
display(pd.crosstab(df_labeled["Label"], df_labeled["Label_binary"]))

Phân bố nhãn nhị phân sau khi gán:


,count
Label_binary,
Benign (0),1496505
Attack (1),281887



Kiểm tra chéo Label gốc vs Label_binary:


Label_binary,0,1
Label,,
Benign,1496505,0
Botnet,0,736
DDoS,0,95144
DoS GoldenEye,0,7567
DoS Hulk,0,158449
DoS Slowhttptest,0,1741
DoS Slowloris,0,3998
FTP-Patator,0,3972
Heartbleed,0,11


### 5. Lưu kết quả để dùng cho các bước tiếp theo

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/TTTN/cic_ids_2017_labeled.parquet"
df_labeled.to_parquet(OUTPUT_PATH, index=False)
print("Đã lưu dataset đã gán nhãn tại:", OUTPUT_PATH)
print("Shape cuối cùng:", df_labeled.shape)

Đã lưu dataset đã gán nhãn tại: /content/drive/MyDrive/TTTN/cic_ids_2017_labeled.parquet
Shape cuối cùng: (1778392, 85)


In [ ]:
pd.set_option("display.max_columns", None)
df_labeled.head(10)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd RST Flags,Bwd RST Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Avg Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Act Data Packets,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label,__source_file,Label_binary
0,0,119719148,231,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,1.929516,5.205180e+05,2274525.75,20694308.0,0.0,119719152.0,5.205180e+05,2274525.75,20694308.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,1.929516,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0.0,0.000000,0.000000,0.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,8721244.0,8761761.0,22509460.0,17.0,12685486.0,5296658.0,20694308.0,6499982.0,-1,-1,0,Benign,Benign-Monday.parquet,0
1,17,65511209,6,6,288.0,288.0,48.0,48.0,48.000000,0.000000,48.0,48.0,48.0,0.0,8.792388,0.183175,5.955564e+06,19258082.00,64004884.0,3.0,65511096.0,1.310222e+07,28462970.00,64004960.0,3.0,65511104.0,13102221.0,28462988.0,64004996.0,3.0,0,0,0,0,0,0,48,48,0.091587,0.091587,48.0,48.0,48.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,1.0,48.000000,48.000000,48.0,0,0,0,0,0,0,0,24,0,24,0,0,5,8,1506210.0,0.0,1506210.0,1506210.0,64004884.0,0.0,64004884.0,64004884.0,-1,-1,0,Benign,Benign-Monday.parquet,0
2,17,113976922,267,0,20447.0,0.0,153.0,37.0,76.580521,44.140625,0.0,0.0,0.0,0.0,179.395966,2.342579,4.284847e+05,3701805.25,48523116.0,0.0,113976920.0,4.284847e+05,3701805.25,48523116.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,2136,0,2.342580,0.000000,37.0,153.0,76.580521,44.140625,1948.394775,0,0,0,0,0,0,0,0,0.0,76.580521,76.580521,0.0,2044,26,4794,0,0,0,1,76,0,0,0,0,266,8,2745981.5,5491934.5,10983883.0,14.0,25498178.0,18833046.0,48523116.0,5463561.0,-1,-1,0,Benign,Benign-Monday.parquet,0
3,17,67037196,8,8,384.0,384.0,48.0,48.0,48.000000,0.000000,48.0,48.0,48.0,0.0,11.456326,0.238673,4.469146e+06,14271655.00,55956316.0,44916.0,66990996.0,9.570142e+06,20477636.00,56001260.0,992873.0,66991088.0,9570156.0,20478190.0,56002516.0,991683.0,0,0,0,0,0,0,64,64,0.119337,0.119337,48.0,48.0,48.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,1.0,48.000000,48.000000,48.0,0,0,0,0,0,0,0,24,0,24,0,0,7,8,11034681.0,0.0,11034681.0,11034681.0,55956316.0,0.0,55956316.0,55956316.0,-1,-1,0,Benign,Benign-Monday.parquet,0
4,17,68045057,8,8,384.0,384.0,48.0,48.0,48.000000,0.000000,48.0,48.0,48.0,0.0,11.286639,0.235138,4.536337e+06,14525054.00,56943904.0,57451.0,67987504.0,9.712500e+06,20855822.00,57001356.0,989295.0,67987528.0,9712504.0,20855866.0,57001460.0,989348.0,0,0,0,0,0,0,64,64,0.117569,0.117569,48.0,48.0,48.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,1.0,48.000000,48.000000,48.0,0,0,0,0,0,0,0,24,0,24,0,0,7,8,11043596.0,0.0,11043596.0,11043596.0,56943904.0,0.0,56943904.0,56943904.0,-1,-1,0,Benign,Benign-Monday.parquet,0
5,17,70053281,3,3,144.0,144.0,48.0,48.0,48.000000,0.000000,48.0,48.0,48.0,0.0,4.111156,0.085649,1.401066e+07,29592712.00,66904116.0,93919.0,69959072.0,3.497954e+07,45281160.00,66998